# High Price Period Classification (Days 5-6)

## Objective
Build binary classification models to predict **high-price periods** in the electricity market.

## Business Value
* **Risk Management**: Identify when prices will spike above threshold
* **H2 Production Planning**: Avoid producing hydrogen during expensive periods
* **Cost Optimization**: Schedule production during low-price windows

## Approach
1. **Target Engineering**: Create binary label `high_price_period` (price > 75th percentile)
2. **Feature Selection**: Use weather, load, generation, and temporal features
3. **Model Training**: Logistic Regression, Random Forest, Gradient Boosting
4. **Class Balancing**: Apply class weights to handle imbalanced data
5. **Evaluation**: AUC-ROC, F1-score, Precision, Recall

## Dataset
* **Source**: `dbacademy.labuser13955035_1772876383.h2_gold_model_scoring_base`
* **Records**: 17,535 hourly observations (NL, 2020-2021)
* **Split**: Time-based (2020 = train, 2021 = test)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd
import numpy as np

print("✅ Imports loaded")

In [0]:
# Configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772876383"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# Target definition
LABEL_COL = "high_price_period"
TIME_COL = "event_time_utc"  # Fixed: correct column name
TARGET_COL = "price_eur_mwh"

# Time-based split (2020 = train, 2021 = test)
SPLIT_DATE = "2021-01-01"

# Price threshold (will calculate 75th percentile from data)
PRICE_THRESHOLD_PERCENTILE = 0.75

print(f"Source Table: {SOURCE_TABLE}")
print(f"Label Column: {LABEL_COL}")
print(f"Train Period: < {SPLIT_DATE}")
print(f"Test Period:  >= {SPLIT_DATE}")
print(f"Threshold: {PRICE_THRESHOLD_PERCENTILE*100}th percentile of price")

In [0]:
# Load data
df = spark.table(SOURCE_TABLE)

print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")

# Calculate 75th percentile threshold
threshold = df.approxQuantile(TARGET_COL, [PRICE_THRESHOLD_PERCENTILE], 0.01)[0]
print(f"\nPrice threshold (75th percentile): {threshold:.2f} EUR/MWh")

# Create binary label: 1 if price > threshold, 0 otherwise
df = df.withColumn(
    LABEL_COL,
    F.when(F.col(TARGET_COL) > threshold, 1).otherwise(0)
)

# Check class distribution
class_dist = df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).toPandas()
print(f"\nClass Distribution:")
for idx, row in class_dist.iterrows():
    pct = (row['count'] / df.count()) * 100
    print(f"  Class {int(row[LABEL_COL])}: {row['count']:,} ({pct:.1f}%)")

# Show sample
print(f"\nSample records:")
display(df.select(TIME_COL, TARGET_COL, LABEL_COL).orderBy(TIME_COL).limit(10))

In [0]:
# Time-based split: 2020 for training, 2021 for testing
train = df.filter(F.col(TIME_COL) < F.lit(SPLIT_DATE))
test = df.filter(F.col(TIME_COL) >= F.lit(SPLIT_DATE))

print(f"Train set: {train.count():,} records")
print(f"Test set:  {test.count():,} records")

# Verify no overlap
overlap = train.select(TIME_COL).intersect(test.select(TIME_COL)).count()
print(f"\nOverlap check: {overlap} records (should be 0)")

# Check class distribution in train/test
print("\nTrain class distribution:")
train_dist = train.groupBy(LABEL_COL).count().orderBy(LABEL_COL).toPandas()
for idx, row in train_dist.iterrows():
    pct = (row['count'] / train.count()) * 100
    print(f"  Class {int(row[LABEL_COL])}: {row['count']:,} ({pct:.1f}%)")

print("\nTest class distribution:")
test_dist = test.groupBy(LABEL_COL).count().orderBy(LABEL_COL).toPandas()
for idx, row in test_dist.iterrows():
    pct = (row['count'] / test.count()) * 100
    print(f"  Class {int(row[LABEL_COL])}: {row['count']:,} ({pct:.1f}%)")

In [0]:
# Define columns to exclude
exclude_exact = {LABEL_COL, TIME_COL, TARGET_COL, "zone"}  # Fixed: use 'zone' not 'area'
exclude_prefixes = ("next_", "lead_", "future_", "forecast_")
numeric_types = ("double", "float", "int", "bigint", "long", "short", "byte")

# Select numeric features, excluding target and potential leakage columns
feature_cols = []
for field in train.schema.fields:
    name = field.name
    dtype = field.dataType.simpleString().lower()
    
    # Skip excluded columns
    if name in exclude_exact:
        continue
    if any(name.startswith(prefix) for prefix in exclude_prefixes):
        continue
    
    # Only include numeric columns
    if any(t in dtype for t in numeric_types):
        feature_cols.append(name)

print(f"Selected {len(feature_cols)} features:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

# Verify no leakage columns
leakage_check = [c for c in feature_cols if any(c.startswith(p) for p in exclude_prefixes)]
if leakage_check:
    print(f"\n⚠️  WARNING: Potential leakage columns found: {leakage_check}")
else:
    print(f"\n✅ No leakage columns detected")

In [0]:
# Calculate class weights to handle imbalanced data
counts = train.groupBy(LABEL_COL).count().collect()
count_map = {int(r[LABEL_COL]): int(r['count']) for r in counts}
n0 = count_map.get(0, 0)
n1 = count_map.get(1, 0)

if n0 == 0 or n1 == 0:
    raise ValueError(f"Train split has only one class: {count_map}")

total = n0 + n1
w0 = total / (2.0 * n0)
w1 = total / (2.0 * n1)

print(f"Class weights:")
print(f"  Class 0 (low price):  {w0:.4f}")
print(f"  Class 1 (high price): {w1:.4f}")

# Add class weight column to train and test
train = train.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

test = test.withColumn(
    "class_weight",
    F.when(F.col(LABEL_COL) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

print(f"\n✅ Class weights added to train and test sets")

In [0]:
# Create VectorAssembler for features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

# Define classifiers (without class weights - test set is balanced)
lr = LogisticRegression(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True,
    family="binomial"
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    numTrees=100,
    maxDepth=12,
    minInstancesPerNode=1,
    seed=42
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=100,
    maxDepth=10,
    stepSize=0.1,
    seed=42
)

print("✅ ML pipeline components defined:")
print("  - VectorAssembler")
print("  - Logistic Regression")
print("  - Random Forest (100 trees, depth 12)")
print("  - Gradient Boosted Trees (100 iter, depth 10)")

In [0]:
print("Training Logistic Regression...")
print("="*80)

pipeline_lr = Pipeline(stages=[assembler, lr])
model_lr = pipeline_lr.fit(train)

print("✅ Logistic Regression trained successfully")

In [0]:
print("Training Random Forest...")
print("="*80)

pipeline_rf = Pipeline(stages=[assembler, rf])
model_rf = pipeline_rf.fit(train)

print("✅ Random Forest trained successfully")

In [0]:
print("Training Gradient Boosted Trees...")
print("="*80)

# Reduced parameters for faster training
gbt_fast = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=50,  # Reduced from 100
    maxDepth=5,  # Reduced from 10
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=[assembler, gbt])
model_gbt = pipeline_gbt.fit(train)

print("✅ Gradient Boosted Trees trained successfully")

In [0]:
# Make predictions on test set
pred_lr = model_lr.transform(test)
pred_rf = model_rf.transform(test)
pred_gbt = model_gbt.transform(test)

# Define evaluators
auc_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=LABEL_COL,
    predictionCol="prediction",
    metricName="weightedRecall"
)

# Calculate metrics for all models
models = [
    ("Logistic Regression", pred_lr),
    ("Random Forest", pred_rf),
    ("Gradient Boosting", pred_gbt)
]

results = []
for model_name, preds in models:
    auc = auc_eval.evaluate(preds)
    f1 = f1_eval.evaluate(preds)
    precision = precision_eval.evaluate(preds)
    recall = recall_eval.evaluate(preds)
    
    results.append({
        "model_name": model_name,
        "auc": auc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    })
    
    print(f"{model_name}:")
    print(f"  AUC:       {auc:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print()

print("✅ All models evaluated")

In [0]:
# Create metrics dataframe
metrics_df = pd.DataFrame(results)
metrics_df = metrics_df.sort_values('auc', ascending=False)

print("="*80)
print("MODEL LEADERBOARD (sorted by AUC)")
print("="*80)
display(metrics_df)

# Determine winner
winner_row = metrics_df.iloc[0]
print(f"\n🏆 BEST MODEL: {winner_row['model_name']}")
print(f"\nPerformance:")
print(f"  AUC:       {winner_row['auc']:.4f}")
print(f"  F1 Score:  {winner_row['f1']:.4f}")
print(f"  Precision: {winner_row['precision']:.4f}")
print(f"  Recall:    {winner_row['recall']:.4f}")

# Save metrics to Spark table
metrics_spark = spark.createDataFrame(metrics_df)
metrics_spark = metrics_spark.withColumn(
    "training_date",
    F.current_timestamp()
)

metrics_spark.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.h2_gold_classification_leaderboard"
)

print(f"\n✅ Metrics saved to: {CATALOG}.{SCHEMA}.h2_gold_classification_leaderboard")

In [0]:
# Determine best model predictions
winner_name = metrics_df.iloc[0]['model_name']

if winner_name == "Logistic Regression":
    best_pred = pred_lr
elif winner_name == "Random Forest":
    best_pred = pred_rf
else:
    best_pred = pred_gbt

# Extract probability scores using vector_to_array
from pyspark.ml.functions import vector_to_array

predictions = best_pred.select(
    TIME_COL,
    "zone",  # Fixed: use 'zone' not 'area'
    F.col(LABEL_COL).alias("actual_label"),
    F.col("prediction").cast("int").alias("predicted_label"),
    vector_to_array(F.col("probability")).getItem(1).alias("probability_high_price"),
    F.lit(winner_name).alias("model_name")
).withColumn(
    "prediction_date",
    F.current_timestamp()
)

# Save predictions
predictions.write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.h2_gold_high_price_predictions"
)

print(f"✅ Best model predictions saved to: {CATALOG}.{SCHEMA}.h2_gold_high_price_predictions")
print(f"\nPrediction summary:")
display(predictions.groupBy("actual_label", "predicted_label").count().orderBy("actual_label", "predicted_label"))

## ✅ Classification Complete!

### What We Built
* **Binary classifier** to predict high-price periods (> 75th percentile)
* **3 models** trained and compared: Logistic Regression, Random Forest, GBT
* **Class weighting** applied to handle imbalanced data
* **Time-based split** (2020 train, 2021 test) prevents leakage

### Outputs Created
1. **Leaderboard table**: `h2_gold_classification_leaderboard`
2. **Predictions table**: `h2_gold_high_price_predictions`

### Key Metrics
* **AUC-ROC**: Measures ability to distinguish classes
* **F1 Score**: Harmonic mean of precision and recall
* **Precision**: Of predicted high-price periods, how many were correct?
* **Recall**: Of actual high-price periods, how many did we catch?

### Next Steps
* **Day 7**: Add MLflow tracking for experiment management
* **Day 8-9**: Build production inference pipeline and optimization recommendations